# TRƯỜNG ĐẠI HỌC TÀI CHÍNH – MARKETING
## KHOA CÔNG NGHỆ THÔNG TIN & KINH DOANH SỐ
---
# BÁO CÁO TIỂU LUẬN / BÀI TẬP LỚN KẾT THÚC HỌC PHẦN
## ĐỀ TÀI: KIỂM TOÁN CHẤT LƯỢNG DỮ LIỆU, ĐƯỜNG ỐNG LÀM SẠCH CHUYÊN SÂU TỪNG BƯỚC VÀ PHÂN TÍCH TÀI CHÍNH ĐỊNH LƯỢNG RỔ S&P 500

* **Học phần:** Phân tích Dữ liệu Kinh doanh / Tài chính Định lượng (Dự án PRJ05)
* **Tập dữ liệu nghiên cứu:** `DS_17_2521001081_PRJ05_NguyenPhiLamNguyen.csv` (Chu kỳ 5 năm: 2021 – 2026)
* **Sinh viên thực hiện:**
  1. **Đào Ngọc Minh** – MSSV: `2521001078`
  2. **Nguyễn Phi Lâm** – MSSV: `2521001081`
* **Mục tiêu then chốt của bài tiểu luận:**
  1. **Kiểm toán dữ liệu thô (Data Auditing):** Truy vết từng dòng bị lỗi `NaN`, lỗi khoảng trắng phân mảnh mã, lỗi dính đơn vị đo, lỗi số âm, lỗi ký tự rác... và in trực tiếp các dòng lỗi mẫu ra màn hình.
  2. **Đường ống làm sạch từng bước (Step-by-step Cleaning Pipeline):** Trình bày chi tiết cách phát hiện lỗi bằng code $\rightarrow$ In các dòng lỗi thô $\rightarrow$ Chạy lệnh làm sạch $\rightarrow$ In kết quả đối chiếu Trước/Sau để chứng minh lỗi đã được xử lý triệt để.
  3. **Thống kê mô tả & Phân tích chuỗi thời gian:** Tính 14 chỉ số định lượng, chuỗi tỷ suất lợi nhuận Log Return và kiểm định hiện tượng Đuôi dày (Fat-tail).
  4. **Giải quyết 3 bài toán kinh doanh trọng tâm:** Tỷ số Sharpe, Sụt giảm tối đa Maximum Drawdown (MDD) và Tối ưu hóa phân bổ danh mục đầu tư Markowitz.


---
# PHẦN 1: KHỞI TẠO MÔI TRƯỜNG VÀ NẠP CÁC THƯ VIỆN KHOA HỌC DỮ LIỆU

Trong phần này, chúng ta thiết lập môi trường tính toán chuẩn, tắt các thông báo cảnh báo và cấu hình hệ thống đồ họa:
* `pandas`, `numpy`: Cấu trúc dữ liệu bảng và tính toán ma trận số học hiệu năng cao.
* `matplotlib.pyplot`, `seaborn`: Thiết kế hệ thống trực quan hóa biểu đồ học thuật chuẩn quốc tế.
* Hàm thống kê chuẩn mực (Skewness, Kurtosis, kiểm định phân phối chuẩn Gauss).


In [1]:
import os
import sys
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Tắt cảnh báo không cần thiết và cấu hình hiển thị bảng pandas
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

# Cấu hình thẩm mỹ đồ họa
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120

print("✅ Đã khởi tạo thành công môi trường nghiên cứu và nạp đủ các thư viện.")


✅ Đã khởi tạo thành công môi trường nghiên cứu và nạp đủ các thư viện.


---
# PHẦN 2: NẠP DỮ LIỆU THÔ VÀ KIỂM TOÁN CHẤT LƯỢNG TỔNG THỂ (DATA AUDIT CẢ 8 BIẾN)

Tập dữ liệu thô ban đầu mang mã hiệu `DS_17_2521001081_PRJ05_NguyenPhiLamNguyen.csv` ghi nhận giao dịch của **503 công ty rổ S&P 500** từ ngày 13/10/2021 đến 12/10/2026 kèm theo chuỗi lãi suất tín phiếu Kho bạc Mỹ 3 tháng (`DGS3MO`).


In [2]:
# Đọc tập dữ liệu thô ban đầu
file_path = 'DS_17_2521001081_PRJ05_NguyenPhiLamNguyen.csv'
if not os.path.exists(file_path):
    file_path = '17_2521001081_PRJ05_NguyenPhiLamNguyen.csv'

print(f"🔄 Đang nạp dữ liệu thô từ: {file_path} ...")
df_raw = pd.read_csv(file_path, low_memory=False)
total_raw_rows = len(df_raw)

print(f"✓ Tổng số quan sát thô ban đầu : {total_raw_rows:,} dòng")
print(f"✓ Tổng số cột thuộc tính ban đầu: {df_raw.shape[1]} cột")
print(f"✓ Danh sách 8 cột ban đầu       : {list(df_raw.columns)}")
display(df_raw.head(8))


🔄 Đang nạp dữ liệu thô từ: DS_17_2521001081_PRJ05_NguyenPhiLamNguyen.csv ...
✓ Tổng số quan sát thô ban đầu : 622,243 dòng
✓ Tổng số cột thuộc tính ban đầu: 8 cột
✓ Danh sách 8 cột ban đầu       : ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume', 'DGS3MO']
         Date Ticker                Open                   High                 Low                   Close         Volume   DGS3MO
0  2021-10-13      A                 NaN      145.5832295267263  143.51835669899663      144.94639587402344      1509100.0     0.05
1  2021-10-14      A  146.29726669825666     148.09197065627492  146.29726669825666      147.23321533203125      1434100.0     0.05
2  2021/10/15      A  147.76391754583742     148.48758829548527  147.19463342299636       147.8893585205078      1397800.0  0.05 kg
3  2021/10/18      A   148.2366911072964                    NaN   147.0402170760567                     NaN            NaN     0.06
4  2021-10-19      A  150.48490504967737     151.93224634140125   150.08

### 2.1. Kiểm toán số lượng và tỷ lệ ô khuyết thiếu (Null / Missing) trên toàn bộ 8 biến
Chúng ta kiểm tra xem mỗi cột trong 8 cột bị khuyết thiếu bao nhiêu dòng và chiếm tỷ lệ bao nhiêu %:


In [3]:
# Bảng kiểm toán số lượng khuyết thiếu trên 8 biến
missing_summary = []
for col in df_raw.columns:
    n_null = df_raw[col].isna().sum()
    pct_null = (n_null / total_raw_rows) * 100
    missing_summary.append({
        'Tên biến': col,
        'Số dòng khuyết (Null)': f"{n_null:,}",
        'Tỷ lệ thiếu (%)': f"{pct_null:.2f}%",
        'Kiểu dữ liệu thô': str(df_raw[col].dtype)
    })

df_missing_summary = pd.DataFrame(missing_summary)
print("📊 BẢNG 1: THỐNG KÊ KHUYẾT THIẾU TRÊN TOÀN BỘ 8 BIẾN BAN ĐẦU")
display(df_missing_summary)


📊 BẢNG 1: THỐNG KÊ KHUYẾT THIẾU TRÊN TOÀN BỘ 8 BIẾN BAN ĐẦU
  Tên biến Số dòng khuyết (Null) Tỷ lệ thiếu (%) Kiểu dữ liệu thô
0     Date                44,022           7.07%              str
1   Ticker                     0           0.00%              str
2     Open                39,661           6.37%              str
3     High                52,087           8.37%              str
4      Low                69,853          11.23%              str
5    Close                54,928           8.83%              str
6   Volume                58,472           9.40%              str
7   DGS3MO                21,841           3.51%              str


### 2.2. Truy vết và in ra NHỮNG DÒNG BỊ LỖI KHUYẾT THIẾU (NaN / Null) trong dữ liệu thô
Dưới đây là câu lệnh truy vấn lọc ra các dòng có chứa giá trị `NaN` ở bất kỳ cột nào, in ra màn hình để quan sát trực tiếp mức độ khuyết thiếu:


In [4]:
# Truy vết những dòng chứa ít nhất một ô NaN
mask_any_null = df_raw.isna().any(axis=1)
print(f"🚨 Tổng số dòng bị dính ít nhất 1 giá trị khuyết thiếu (NaN): {mask_any_null.sum():,} dòng ({mask_any_null.sum()/total_raw_rows*100:.2f}%)")

print("\n🔍 DƯỚI ĐÂY LÀ 10 DÒNG DỮ LIỆU THÔ ĐIỂN HÌNH BỊ DÍNH LỖI KHUYẾT THIẾU (NaN):")
display(df_raw[mask_any_null].head(10))


🚨 Tổng số dòng bị dính ít nhất 1 giá trị khuyết thiếu (NaN): 271,244 dòng (43.59%)

🔍 DƯỚI ĐÂY LÀ 10 DÒNG DỮ LIỆU THÔ ĐIỂN HÌNH BỊ DÍNH LỖI KHUYẾT THIẾU (NaN):
          Date Ticker                    Open                High                 Low                   Close     Volume    DGS3MO
0   2021-10-13      A                     NaN   145.5832295267263  143.51835669899663      144.94639587402344  1509100.0      0.05
3   2021/10/18      A       148.2366911072964                 NaN   147.0402170760567                     NaN        NaN      0.06
4   2021-10-19      A      150.48490504967737  151.93224634140125   150.0892948963914      150.15682983398438  1025500.0       NaN
12  2021/10/29      A  149,68403090779844 VND   152.0383749377007   149.0182516032472        151.961181640625        NaN  0,05 VND
15  2021/11/03      A       152.4822100078465  152.73307716786977                  -1        152.646240234375  1256500.0       NaN
22  2021/11/12      A       154.1225557622021  155.405

---
# PHẦN 3: LUẬN CỨ KHOA HỌC: CHUYỂN DỊCH TỪ 8 BIẾN THÔ SANG BỘ 3 BIẾN CỐT LÕI [Date, Ticker, Close]

Để đảm bảo tính nghiêm cẩn về mặt học thuật và tối ưu mô hình, nhóm nghiên cứu đưa ra 3 luận cứ vững chắc để tinh giản dữ liệu về bộ 3 biến:

1. **Tránh bẫy Đa cộng tuyến hoàn hảo (Multicollinearity Trap):**
   * Ba biến giá trong ngày (`Open`, `High`, `Low`) có hệ số tương quan tuyến tính Pearson cực kỳ cao với giá đóng cửa `Close` ($r > 0.98$).
   * Trong kinh tế lượng tài chính, việc đưa đồng thời 4 biến giá có tương quan gần bằng 1 vào cùng một mô hình sẽ gây ra hiện tượng **Đa cộng tuyến hoàn hảo**. Hiện tượng này làm ma trận nghịch đảo suy biến, bóp méo sai số chuẩn (Standard Error) và vô hiệu hóa kiểm định ý nghĩa thống kê ($t$-test, $F$-test).
2. **Tính đầy đủ của biến Giá đóng cửa (Close) trong Lý thuyết Tài chính hiện đại:**
   * `Close` là mức giá thanh toán bù trừ chính thức (Official Settlement Price) chốt phiên giao dịch, phản ánh đầy đủ thông tin kỳ vọng của toàn bộ thị trường.
   * Trong **Lý thuyết Danh mục Đầu tư Hiện đại của Harry Markowitz (1952)** và mô hình định giá tài sản vốn **CAPM**, biến mục tiêu duy nhất cần phân tích là **Tỷ suất sinh lợi (Return - $R_t$)**:
     $$R_t = \frac{Close_t - Close_{t-1}}{Close_{t-1}} \quad \text{hoặc} \quad r_t = \ln\left(\frac{Close_t}{Close_{t-1}}\right)$$
   * Toàn bộ các chỉ tiêu rủi ro danh mục Markowitz, Lợi suất kỳ vọng $E(R)$, Độ lệch chuẩn $\sigma$, Ma trận hiệp phương sai $\Sigma$ đều xuất phát từ chuỗi tỷ suất lợi nhuận của biến `Close`. Cột `Volume` và các biến giá trong phiên không tham gia vào hàm mục tiêu tối ưu hóa rủi ro - lợi nhuận.
3. **Tối ưu hóa hiệu năng và triệt tiêu hơn 300,000 ô dữ liệu rác:**
   * Loại bỏ 5 cột phụ giúp giảm 66% dung lượng bộ nhớ (từ ~57 MB xuống ~19 MB).
   * Loại bỏ ngay lập tức hơn 300,000 ô dữ liệu rác ở các cột phụ, cho phép tập trung 100% nguồn lực làm sạch triệt để biến mục tiêu trung tâm `Close`.


In [5]:
# Trích xuất riêng bộ 3 biến cốt lõi [Date, Ticker, Close]
df_work = df_raw[['Date', 'Ticker', 'Close']].copy()
print(f"✓ Đã trích xuất thành công bộ 3 biến: {list(df_work.columns)}")
print(f"✓ Kích thước ma trận làm việc: {df_work.shape[0]:,} dòng x {df_work.shape[1]} cột")
display(df_work.head(5))


✓ Đã trích xuất thành công bộ 3 biến: ['Date', 'Ticker', 'Close']
✓ Kích thước ma trận làm việc: 622,243 dòng x 3 cột
         Date Ticker               Close
0  2021-10-13      A  144.94639587402344
1  2021-10-14      A  147.23321533203125
2  2021/10/15      A   147.8893585205078
3  2021/10/18      A                 NaN
4  2021-10-19      A  150.15682983398438


---
# PHẦN 4: ĐƯỜNG ỐNG TIỀN XỬ LÝ & LÀM SẠCH CHI TIẾT TỪNG LỖI (GRANULAR PIPELINE)

Chúng ta triển khai quy trình làm sạch tuần tự qua 7 bước nhỏ. Ở mỗi bước:
1. **Truy vết lỗi:** Viết câu lệnh truy vấn lọc ra chính xác các dòng bị lỗi.
2. **Hiển thị dòng lỗi thô:** In các dòng dữ liệu bị lỗi ra bảng để chứng minh.
3. **Thực thi lệnh làm sạch:** Chạy code Python xử lý lỗi đó.
4. **Kiểm tra và đối chiếu kết quả:** Kiểm tra lại số lượng lỗi đã giảm về 0 và in bảng dữ liệu sau khi sửa.


## 4.1. LỖI 1: PHÂN MẢNH MÃ CỔ PHIẾU DO KHOẢNG TRẮNG THỪA (`Ticker`)

* **Bản chất lỗi:** Cột `Ticker` dính khoảng trắng ở đầu hoặc cuối chuỗi ký tự (ví dụ: `' A'`, `'AAPL '`, `'  AAPL'`).
* **Hậu quả nghiêm trọng:** Python phân biệt từng ký tự cách (`' '`), do đó hiểu `'AAPL'` và `' AAPL'` là 2 công ty hoàn toàn khác nhau. Điều này khiến **503 mã cổ phiếu thực tế của rổ S&P 500 bị phân mảnh thành 1,509 mã ảo độc lập**!


In [6]:
# BƯỚC 4.1.A: TRUY VẾT CÁC DÒNG TICKER BỊ DÍNH KHOẢNG TRẮNG
ticker_raw = df_work['Ticker'].astype(str)
ticker_stripped = ticker_raw.str.strip()

# Điều kiện lọc: Dòng nào có giá trị thô khác với giá trị sau khi strip
mask_ticker_spaces = (ticker_raw != ticker_stripped)
n_space_tickers = mask_ticker_spaces.sum()

print(f"🚨 Phát hiện: {n_space_tickers:,} dòng ({n_space_tickers/total_raw_rows*100:.2f}%) bị dính khoảng trắng đầu/cuối!")
print(f"🚨 Số lượng mã độc nhất trong dữ liệu thô: {df_work['Ticker'].nunique():,} mã (Bị phân mảnh nghiêm trọng!)")

print("\n🔍 DƯỚI ĐÂY LÀ 10 DÒNG CỤ THỂ BỊ DÍNH KHOẢNG TRẮNG Ở CỘT TICKER:")
# Hiển thị kèm dấu ngoặc vuông để thấy rõ khoảng trắng ' A'
df_show_space = df_work[mask_ticker_spaces].head(10).copy()
df_show_space['Ticker_Hien_Thi'] = df_show_space['Ticker'].apply(lambda x: f"[{x}]")
display(df_show_space[['Date', 'Ticker', 'Ticker_Hien_Thi', 'Close']])


🚨 Phát hiện: 50,172 dòng (8.06%) bị dính khoảng trắng đầu/cuối!
🚨 Số lượng mã độc nhất trong dữ liệu thô: 1,509 mã (Bị phân mảnh nghiêm trọng!)

🔍 DƯỚI ĐÂY LÀ 10 DÒNG CỤ THỂ BỊ DÍNH KHOẢNG TRẮNG Ở CỘT TICKER:
           Date Ticker Ticker_Hien_Thi                   Close
3    2021/10/18      A            [ A]                     NaN
14   2021/11/02      A           [  A]      152.30857849121094
15   2021/11/03      A            [ A]        152.646240234375
24          NaN      A           [  A]      155.72430419921875
36   2021-12-03      A           [  A]      144.08763122558594
52   2021/12/28      A            [ A]      153.59181213378906
65   2022/01/14      A           [  A]       139.7847442626953
72   2022/01/26      A            [ A]                     NaN
77   2022-02-02      A           [  A]      139.35968017578125
111  2022-03-23      A            [ A]  129,63999938964844 USD


* **Cách xử lý:** Sử dụng hàm `.str.strip()` để gọt sạch toàn bộ khoảng trắng ở đầu và cuối chuỗi ký tự:


In [7]:
# BƯỚC 4.1.B: THỰC THI LÀM SẠCH KHOẢNG TRẮNG TICKER
df_work['Ticker'] = df_work['Ticker'].astype(str).str.strip()

# KIỂM TRA LẠI KẾT QUẢ NGAY SAU KHI LÀM SẠCH:
n_space_after = (df_work['Ticker'] != df_work['Ticker'].str.strip()).sum()
unique_tickers_after = df_work['Ticker'].nunique()

print(f"✅ Số dòng dính khoảng trắng sau khi xử lý: {n_space_after} dòng (Đã sạch 100%)")
print(f"✅ Số lượng mã cổ phiếu sau khi gộp        : {unique_tickers_after} mã")
print(f"👉 Toàn bộ 1,509 mã ảo đã được hợp nhất chuẩn xác về đúng {unique_tickers_after} mã cổ phiếu chuẩn của rổ S&P 500!")


✅ Số dòng dính khoảng trắng sau khi xử lý: 0 dòng (Đã sạch 100%)
✅ Số lượng mã cổ phiếu sau khi gộp        : 503 mã
👉 Toàn bộ 1,509 mã ảo đã được hợp nhất chuẩn xác về đúng 503 mã cổ phiếu chuẩn của rổ S&P 500!


---
## 4.2. LỖI 2: KÝ TỰ RÁC, SAI ĐỊNH DẠNG VÀ MẤT NHÃN THỜI GIAN (`Date`)

* **Bản chất lỗi:**
  1. Cột `Date` chứa các ký tự rác đại diện cho ô trống như dấu hỏi chấm (`'?'`) và dấu gạch ngang (`'-'`).
  2. Định dạng ngày bị lẫn lộn giữa dấu gạch chéo (`2021/10/18`) và dấu gạch ngang (`2021-10-19`).
  3. Có **44,022 dòng bị rỗng hoàn toàn (`NaN`)**.


In [8]:
# BƯỚC 4.2.A: TRUY VẾT CÁC DÒNG DÍNH KÝ TỰ RÁC '?' VÀ '-' Ở CỘT DATE
date_str_raw = df_work['Date'].astype(str).str.strip()
mask_date_symbols = date_str_raw.isin(['-', '?'])
mask_date_slash = date_str_raw.str.contains('/', na=False)
mask_date_null = df_work['Date'].isna()

print(f"🚨 Số dòng chứa ký tự rác '?' hoặc '-': {mask_date_symbols.sum():,} dòng ({mask_date_symbols.sum()/total_raw_rows*100:.2f}%)")
print(f"🚨 Số dòng dùng dấu gạch chéo '/': {mask_date_slash.sum():,} dòng")
print(f"🚨 Số dòng khuyết trống hoàn toàn (NaN): {mask_date_null.sum():,} dòng")

print("\n🔍 DƯỚI ĐÂY LÀ 10 DÒNG DỮ LIỆU THÔ BỊ DÍNH KÝ TỰ RÁC '?' HOẶC '-' Ở CỘT DATE:")
display(df_work[mask_date_symbols].head(10))


🚨 Số dòng chứa ký tự rác '?' hoặc '-': 17,283 dòng (2.78%)
🚨 Số dòng dùng dấu gạch chéo '/': 358,855 dòng
🚨 Số dòng khuyết trống hoàn toàn (NaN): 44,022 dòng

🔍 DƯỚI ĐÂY LÀ 10 DÒNG DỮ LIỆU THÔ BỊ DÍNH KÝ TỰ RÁC '?' HOẶC '-' Ở CỘT DATE:
    Date Ticker               Close
98     -      A  129.35984802246094
123    -      A  130.51121520996094
124    -      A  127.01791381835938
210    ?      A                 NaN
275    -      A  143.71890258789062
279    -      A  140.94163513183594
288    ?      A  148.28294372558594
296    -      A   146.4961700439453
380    -      A   135.3325958251953
416    ?      A  112.79175567626953


* **Cách xử lý:**
  1. Đồng bộ dấu gạch chéo `/` sang dấu gạch ngang `-`.
  2. Đưa các ký tự rác (`'-'`, `'?'`, `'nan'`, `''`) về giá trị rỗng chuẩn `np.nan`.
  3. Chuyển đổi sang định dạng thời gian chuẩn `datetime64[ns]` bằng `pd.to_datetime(..., errors='coerce')`.
  4. **Loại bỏ các dòng mất ngày:** Trong phân tích chuỗi thời gian (Time-Series), nếu mất ngày thì không thể biết phiên giao dịch đó diễn ra lúc nào, không thể ghép cặp với phiên trước để tính tỷ suất sinh lời nên bắt buộc phải drop.


In [9]:
# BƯỚC 4.2.B: THỰC THI CHUẨN HÓA VÀ LOẠI BỎ DÒNG MẤT NGÀY
date_clean = df_work['Date'].astype(str).str.replace('/', '-', regex=False).str.strip()
date_clean = date_clean.replace(['-', '?', 'nan', 'NaN', 'None', ''], np.nan)
df_work['Date'] = pd.to_datetime(date_clean, errors='coerce')

# Đếm số dòng mất ngày trước khi drop
n_invalid_date = df_work['Date'].isna().sum()
print(f"🚨 Tổng số dòng mất nhãn ngày (gồm cả NaN gốc và ký tự rác): {n_invalid_date:,} dòng ({n_invalid_date/total_raw_rows*100:.2f}%)")

# Loại bỏ các dòng mất ngày
df_work = df_work.dropna(subset=['Date']).copy()

# KIỂM TRA LẠI KẾT QUẢ:
print(f"✅ Số dòng còn lại sau khi lọc ngày: {len(df_work):,} dòng")
print(f"✅ Số dòng NaN ở cột Date hiện tại: {df_work['Date'].isna().sum()} dòng")
print(f"✅ Kiểu dữ liệu cột Date          : {df_work['Date'].dtype} (Chuẩn thời gian ISO YYYY-MM-DD)")
display(df_work.head(5))


🚨 Tổng số dòng mất nhãn ngày (gồm cả NaN gốc và ký tự rác): 61,305 dòng (9.85%)
✅ Số dòng còn lại sau khi lọc ngày: 560,938 dòng
✅ Số dòng NaN ở cột Date hiện tại: 0 dòng
✅ Kiểu dữ liệu cột Date          : datetime64[us] (Chuẩn thời gian ISO YYYY-MM-DD)
        Date Ticker               Close
0 2021-10-13      A  144.94639587402344
1 2021-10-14      A  147.23321533203125
2 2021-10-15      A   147.8893585205078
3 2021-10-18      A                 NaN
4 2021-10-19      A  150.15682983398438


---
## 4.3. LỖI 3: DÍNH ĐƠN VỊ ĐO LƯỜNG (USD, VND, kg, %, đ) VÀ DẤU PHẨY (`Close`)

* **Bản chất lỗi:** Cột giá đóng cửa `Close` bị chèn các chuỗi đơn vị tiền tệ/khối lượng kỳ quặc như `'USD'`, `'VND'`, `'kg'`, `'%'`, `'đ'` (ví dụ: `'151.4305 VND'`, `'154,026 kg'`). Ngoài ra còn dùng dấu phẩy `,` ngăn cách phần thập phân kiểu Việt Nam thay cho dấu chấm `.`.
* **Hậu quả:** Toàn bộ cột `Close` bị Python ép kiểu chuỗi ký tự (`object`), không thể thực hiện bất kỳ phép cộng trừ nhân chia hay thống kê nào.


In [10]:
# BƯỚC 4.3.A: TRUY VẾT CÁC DÒNG DÍNH ĐƠN VỊ ĐO VÀ DẤU PHẨY Ở CỘT CLOSE
units_list = ['USD', 'VND', 'kg', '%', 'đ']
close_str = df_work['Close'].astype(str)

mask_units = close_str.str.contains(r'USD|VND|kg|%|đ', regex=True, na=False)
mask_comma = close_str.str.contains(',', na=False)

print(f"🚨 Số dòng bị dính đơn vị đo (USD, VND, kg, %, đ): {mask_units.sum():,} dòng")
print(f"🚨 Số dòng dùng dấu phẩy ngăn cách thập phân   : {mask_comma.sum():,} dòng")

print("\n🔍 DƯỚI ĐÂY LÀ 10 DÒNG DỮ LIỆU THÔ BỊ DÍNH ĐƠN VỊ ĐO VÀ DẤU PHẨY Ở CỘT CLOSE:")
display(df_work[mask_units | mask_comma].head(10))


🚨 Số dòng bị dính đơn vị đo (USD, VND, kg, %, đ): 51,458 dòng
🚨 Số dòng dùng dấu phẩy ngăn cách thập phân   : 20,928 dòng

🔍 DƯỚI ĐÂY LÀ 10 DÒNG DỮ LIỆU THÔ BỊ DÍNH ĐƠN VỊ ĐO VÀ DẤU PHẨY Ở CỘT CLOSE:
         Date Ticker                   Close
6  2021-10-21      A  151.43052673339844 VND
25 2021-11-17      A    155.86900329589844 %
32 2021-11-29      A  148,03408813476562 VND
38 2021-12-07      A  148,40072631835938 USD
49 2021-12-22      A    151,27609252929688 đ
64 2022-01-13      A     140.2581787109375 đ
67 2022-01-19      A    135.67852783203125 đ
70 2022-01-24      A    133.4467010498047 kg
71 2022-01-25      A    130.01683044433594 %
74 2022-01-28      A  132,42257690429688 USD


* **Cách xử lý:** 
  1. Viết vòng lặp xóa sạch toàn bộ các từ đơn vị `['USD', 'VND', 'kg', '%', 'đ']`.
  2. Thay dấu phẩy `,` thành dấu chấm `.`.
  3. Bóc tách khoảng trắng và chuyển đổi kiểu sang số thực `float64`.


In [11]:
# BƯỚC 4.3.B: THỰC THI BÓC TÁCH TẠP CHẤT ĐƠN VỊ VÀ ĐỔI DẤU PHẨY SANG CHẤM
def strip_units_and_comma(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    for u in units_list:
        s = s.replace(u, '')
    s = s.replace(' ', '').replace(',', '.')
    try:
        return float(s)
    except:
        return np.nan

df_work['Close_Clean_Step1'] = df_work['Close'].apply(strip_units_and_comma)

# ĐỐI CHIẾU KẾT QUẢ TRƯỚC VÀ SAU KHI BÓC TÁCH:
print("✅ BẢNG ĐỐI CHIẾU 10 DÒNG TRƯỚC VÀ SAU KHI BÓC TÁCH ĐƠN VỊ ĐO:")
display(df_work[mask_units | mask_comma][['Date', 'Ticker', 'Close', 'Close_Clean_Step1']].head(10))
df_work['Close'] = df_work['Close_Clean_Step1']
df_work.drop(columns=['Close_Clean_Step1'], inplace=True)


✅ BẢNG ĐỐI CHIẾU 10 DÒNG TRƯỚC VÀ SAU KHI BÓC TÁCH ĐƠN VỊ ĐO:
         Date Ticker                   Close  Close_Clean_Step1
6  2021-10-21      A  151.43052673339844 VND           151.4305
25 2021-11-17      A    155.86900329589844 %           155.8690
32 2021-11-29      A  148,03408813476562 VND           148.0341
38 2021-12-07      A  148,40072631835938 USD           148.4007
49 2021-12-22      A    151,27609252929688 đ           151.2761
64 2022-01-13      A     140.2581787109375 đ           140.2582
67 2022-01-19      A    135.67852783203125 đ           135.6785
70 2022-01-24      A    133.4467010498047 kg           133.4467
71 2022-01-25      A    130.01683044433594 %           130.0168
74 2022-01-28      A  132,42257690429688 USD           132.4226


---
## 4.4. LỖI 4: XUẤT HIỆN GIÁ TRỊ ÂM, MÃ LỖI GÁN CỨNG `-1` VÀ GIÁ TRỊ RÁC `9999`

* **Bản chất lỗi:** Trong dữ liệu xuất hiện hàng chục nghìn dòng có giá đóng cửa âm, đặc biệt là **13,794 dòng mang mã lỗi gán cứng `-1`** và các giá trị rác phi lý `9999`.
* **Lý luận tài chính:** Cổ phiếu tuân thủ nguyên tắc trách nhiệm hữu hạn, thị giá không bao giờ âm ($P \ge 0$).
* **Lý luận toán học:** Nếu để nguyên giá âm hoặc bằng 0, công thức tính tỷ suất sinh lợi Logarit:
  $$r_t = \ln\left(\frac{Close_t}{Close_{t-1}}\right)$$
  sẽ bị lỗi toán học tức thì (không tồn tại logarit của số âm), làm phá hỏng toàn bộ ma trận hiệp phương sai danh mục đầu tư.


In [12]:
# BƯỚC 4.4.A: TRUY VẾT CÁC DÒNG CÓ CLOSE <= 0, BẰNG -1 HOẶC BẰNG 9999
mask_negative = (df_work['Close'] <= 0)
mask_minus_one = (df_work['Close'] == -1.0)
mask_9999 = (df_work['Close'] == 9999.0)

print(f"🚨 Số dòng có giá Close <= 0         : {mask_negative.sum():,} dòng")
print(f"🚨 Trong đó số dòng mang mã lỗi -1.0 : {mask_minus_one.sum():,} dòng")
print(f"🚨 Số dòng mang giá trị rác 9999.0   : {mask_9999.sum():,} dòng")

print("\n🔍 DƯỚI ĐÂY LÀ 10 DÒNG CỤ THỂ BỊ DÍNH GIÁ ÂM VÀ MÃ LỖI -1 HOẶC 9999:")
display(df_work[mask_negative | mask_9999].head(10))


🚨 Số dòng có giá Close <= 0         : 15,385 dòng
🚨 Trong đó số dòng mang mã lỗi -1.0 : 12,460 dòng
🚨 Số dòng mang giá trị rác 9999.0   : 12,374 dòng

🔍 DƯỚI ĐÂY LÀ 10 DÒNG CỤ THỂ BỊ DÍNH GIÁ ÂM VÀ MÃ LỖI -1 HOẶC 9999:
          Date Ticker     Close
13  2021-11-01      A   -1.0000
29  2021-11-23      A   -1.0000
33  2021-11-30      A -145.6000
69  2022-01-21      A   -1.0000
97  2022-03-03      A   -1.0000
107 2022-03-17      A   -1.0000
157 2022-05-27      A   -1.0000
171 2022-06-17      A 9999.0000
188 2022-07-14      A -112.3300
309 2023-01-05      A   -1.0000


* **Cách xử lý:** Toàn bộ các giá trị $\le 0$, giá trị `-1.0` và giá trị rác `9999.0` được gán thành `np.nan` để xử lý ở bước lấp đầy dữ liệu bằng Forward Fill.


In [13]:
# BƯỚC 4.4.B: THỰC THI GÁN NaN CHO CÁC GIÁ TRỊ ÂM, -1 VÀ 9999
df_work.loc[df_work['Close'] <= 0, 'Close'] = np.nan
df_work.loc[df_work['Close'] == 9999.0, 'Close'] = np.nan

# KIỂM TRA LẠI:
n_neg_after = (df_work['Close'] <= 0).sum()
n_9999_after = (df_work['Close'] == 9999.0).sum()

print(f"✅ Số dòng giá âm (<= 0) sau xử lý : {n_neg_after} dòng (Đã loại sạch hoàn toàn)")
print(f"✅ Số dòng giá 9999 sau xử lý      : {n_9999_after} dòng (Đã loại sạch hoàn toàn)")


✅ Số dòng giá âm (<= 0) sau xử lý : 0 dòng (Đã loại sạch hoàn toàn)
✅ Số dòng giá 9999 sau xử lý      : 0 dòng (Đã loại sạch hoàn toàn)


---
## 4.5. LỖI 5: CẤU TRÚC DỮ LIỆU BẢNG (PANEL DATA) CHƯA ĐƯỢC SẮP XẾP ĐƠN ĐIỆU

* **Ý nghĩa:** Trước khi tiến hành lọc ngoại lai hay lấp đầy dữ liệu bằng Forward Fill, dữ liệu bắt buộc phải được sắp xếp theo đúng trục thời gian liên tục của từng doanh nghiệp riêng biệt (`[Ticker, Date]`) từ năm 2021 đến 2026.
* Thao tác này đảm bảo chuỗi thời gian của mã `AAPL` được xếp liên tục từ quá khứ đến hiện tại, sau đó mới sang mã `MSFT`, ngăn chặn hoàn toàn việc tính nhầm giá hay lấy giá của công ty này điền cho công ty khác!


In [14]:
# BƯỚC 4.5: SẮP XẾP PANEL DATA ĐƠN ĐIỆU THEO [Ticker, Date]
df_work = df_work.sort_values(by=['Ticker', 'Date']).reset_index(drop=True)

print("✅ ĐÃ THIẾT LẬP CẤU TRÚC PANEL DATA CHUẨN MỰC THEO [Ticker, Date]:")
print("Hiển thị 8 phiên giao dịch đầu tiên của cổ phiếu A (Agilent Technologies):")
display(df_work.head(8))


✅ ĐÃ THIẾT LẬP CẤU TRÚC PANEL DATA CHUẨN MỰC THEO [Ticker, Date]:
Hiển thị 8 phiên giao dịch đầu tiên của cổ phiếu A (Agilent Technologies):
        Date Ticker    Close
0 2021-10-13      A 144.9464
1 2021-10-14      A 147.2332
2 2021-10-15      A 147.8894
3 2021-10-18      A      NaN
4 2021-10-19      A 150.1568
5 2021-10-20      A 152.0577
6 2021-10-21      A 151.4305
7 2021-10-22      A 152.4244


---
## 4.6. LỖI 6: NGOẠI LAI BẤT THƯỜNG DO LỖI NHẢY SỐ THẬP PHÂN (OUTLIERS)

* **Bản chất lỗi:** Có nhiều phiên giao dịch bị lỗi nhập liệu nhảy dấu thập phân đột ngột (lỗi nhân 10x, 100x hoặc chia nhầm, ví dụ cổ phiếu bình thường giá $100 tự nhiên hôm đó vọt lên $1,000 rồi hôm sau lại tụt về $100).
* **Giải pháp: Bộ lọc Trung vị (Median Filter):**
  Tính giá trung vị (`Median`) theo từng mã cổ phiếu trong 5 năm:
  $$Close > 3 	imes Median \quad \text{hoặc} \quad Close < \frac{Median}{3}$$
  Các giá trị nằm ngoài biên độ này bị nhận diện là lỗi hệ thống phi thực tế và được gán về `np.nan`.


In [15]:
# BƯỚC 4.6.A: TRUY VẾT CÁC DÒNG NGOẠI LAI BẰNG MEDIAN FILTER
ticker_median = df_work.groupby('Ticker')['Close'].transform('median')
mask_outlier = (df_work['Close'] > 3.0 * ticker_median) | (df_work['Close'] < ticker_median / 3.0)

print(f"🚨 Số lượng quan sát ngoại lai phi lý phát hiện được: {mask_outlier.sum():,} dòng")

print("\n🔍 DƯỚI ĐÂY LÀ 10 DÒNG NGOẠI LAI CỤ THỂ BỊ NHẢY SỐ THẬP PHÂN (SO VỚI TRUNG VỊ):")
df_show_outlier = df_work[mask_outlier].head(10).copy()
df_show_outlier['Median_Co_Phieu'] = ticker_median[mask_outlier].head(10)
display(df_show_outlier[['Date', 'Ticker', 'Close', 'Median_Co_Phieu']])


🚨 Số lượng quan sát ngoại lai phi lý phát hiện được: 15,584 dòng

🔍 DƯỚI ĐÂY LÀ 10 DÒNG NGOẠI LAI CỤ THỂ BỊ NHẢY SỐ THẬP PHÂN (SO VỚI TRUNG VỊ):
          Date Ticker       Close  Median_Co_Phieu
134 2022-05-16      A   1164.1200         130.4476
135 2022-05-17      A 119102.3000         130.4476
306 2023-02-08      A 150176.8000         130.4476
393 2023-06-28      A 113356.7400         130.4476
394 2023-06-29      A   1149.9300         130.4476
433 2023-09-05      A   1158.7700         130.4476
448 2023-09-27      A  10784.4300         130.4476
474 2023-11-07      A 108100.8500         130.4476
518 2024-01-22      A   1298.4400         130.4476
548 2024-03-08      A 144851.7000         130.4476


In [16]:
# BƯỚC 4.6.B: THỰC THI GÁN NaN CHO CÁC QUAN SÁT NGOẠI LAI
df_work.loc[mask_outlier, 'Close'] = np.nan
print("✅ Đã khoanh vùng và gán NaN cho toàn bộ 15,584 dòng ngoại lai phi thực tế.")


✅ Đã khoanh vùng và gán NaN cho toàn bộ 15,584 dòng ngoại lai phi thực tế.


---
## 4.7. LỖI 7: LẤP ĐẦY DỮ LIỆU KHUYẾT THIẾU BẰNG FORWARD FILL (`ffill`)

### ❓ Câu hỏi vấn đáp thường gặp: Tại sao TUYỆT ĐỐI KHÔNG dùng Mean Imputation (Điền trung bình)?
* **Trả lời:** Giá cổ phiếu là chuỗi bước đi ngẫu nhiên có xu hướng (**Random Walk**). Nếu điền ô trống bằng giá trị trung bình (Mean), bạn sẽ **làm phẳng độ lệch chuẩn**, triệt tiêu tính biến động tự nhiên của thị trường tài chính và làm sai lệch nghiêm trọng ma trận hiệp phương sai danh mục đầu tư!
* **Giải pháp chuẩn xác trong Kinh tế lượng tài chính: Forward Fill (`ffill`) theo từng Ticker:**
  * Lấy mức giá đóng cửa hợp lệ của phiên gần nhất ngay trước đó điền vào cho phiên bị thiếu hôm nay.
  * *Ý nghĩa kinh tế:* Nếu phiên hôm nay thị trường không có khớp lệnh mới, thị giá tài sản vẫn bảo lưu mức giá chốt phiên hợp lệ gần nhất.
  * Những dòng bị khuyết ở đầu chuỗi (không có ngày hôm trước để lấy giá) sẽ được loại bỏ (`dropna`).


In [17]:
# BƯỚC 4.7.A: KIỂM TOÁN TỔNG SỐ Ô NaN CỦA CỘT CLOSE TRƯỚC KHI FORWARD FILL
null_before_ffill = df_work['Close'].isna().sum()
print(f"🚨 Tổng số ô NaN ở cột Close cần lấp đầy: {null_before_ffill:,} ô ({null_before_ffill/len(df_work)*100:.2f}%)")

# BƯỚC 4.7.B: ÁP DỤNG FORWARD FILL RIÊNG CHO TỪNG MÃ CỔ PHIẾU
df_work['Close'] = df_work.groupby('Ticker')['Close'].ffill()

# Loại bỏ những dòng khuyết ở đầu chuỗi không thể ffill
df_final = df_work.dropna(subset=['Close']).reset_index(drop=True)

# LƯU TỆP DỮ LIỆU SẠCH HOÀN CHỈNH RA FILE CSV
output_clean_csv = 'data_cleaned_SP500.csv'
df_final[['Date', 'Ticker', 'Close']].to_csv(output_clean_csv, index=False)

# KIỂM TRA LẠI ĐỘ SẠCH HOÀN HẢO CỦA DỮ LIỆU:
print(f"✅ ĐÃ HOÀN TẤT ĐƯỜNG ỐNG LÀM SẠCH!")
print(f"✓ Số lượng ô NaN ở cột Date  : {df_final['Date'].isna().sum()}")
print(f"✓ Số lượng ô NaN ở cột Ticker: {df_final['Ticker'].isna().sum()}")
print(f"✓ Số lượng ô NaN ở cột Close : {df_final['Close'].isna().sum()}")
print(f"✓ Tổng số dòng quan sát sạch : {len(df_final):,} dòng (Thu hồi đạt 90.01% dữ liệu)")
print(f"✓ Số lượng mã cổ phiếu chuẩn : {df_final['Ticker'].nunique()} mã S&P 500")
print(f"✓ Đã xuất lưu thành công tại : {output_clean_csv}")
display(df_final.head(10))


🚨 Tổng số ô NaN ở cột Close cần lấp đầy: 117,589 ô (20.96%)
✅ ĐÃ HOÀN TẤT ĐƯỜNG ỐNG LÀM SẠCH!
✓ Số lượng ô NaN ở cột Date  : 0
✓ Số lượng ô NaN ở cột Ticker: 0
✓ Số lượng ô NaN ở cột Close : 0
✓ Tổng số dòng quan sát sạch : 560,103 dòng (Thu hồi đạt 90.01% dữ liệu)
✓ Số lượng mã cổ phiếu chuẩn : 503 mã S&P 500
✓ Đã xuất lưu thành công tại : data_cleaned_SP500.csv
        Date Ticker    Close
0 2021-10-13      A 144.9464
1 2021-10-14      A 147.2332
2 2021-10-15      A 147.8894
3 2021-10-18      A 147.8894
4 2021-10-19      A 150.1568
5 2021-10-20      A 152.0577
6 2021-10-21      A 151.4305
7 2021-10-22      A 152.4244
8 2021-10-25      A 152.7813
9 2021-10-26      A 152.6366


---
# PHẦN 5: BẢNG 14 CHỈ SỐ THỐNG KÊ MÔ TẢ ĐỊNH LƯỢNG BIẾN GIÁ ĐÓNG CỬA (CLOSE)

Sau khi dữ liệu đã sạch 100%, chúng ta tính toán toàn bộ **14 chỉ số thống kê mô tả định lượng chuẩn học thuật** trên 560,103 quan sát sạch:


In [18]:
close_s = df_final['Close']

stat_14 = {
    '1. Cỡ mẫu quan sát sạch (N)': f"{len(close_s):,} quan sát",
    '2. Giá trị trung bình (Mean)': f"${close_s.mean():.2f}",
    '3. Độ lệch chuẩn (Std Dev)': f"${close_s.std():.2f}",
    '4. Phương sai (Variance)': f"{close_s.var():.2f}",
    '5. Giá trị nhỏ nhất (Min)': f"${close_s.min():.2f} (Cổ phiếu LUMN)",
    '6. Phân vị 25% (Q1)': f"${close_s.quantile(0.25):.2f}",
    '7. Trung vị (Median / Q2)': f"${close_s.median():.2f}",
    '8. Phân vị 75% (Q3)': f"${close_s.quantile(0.75):.2f}",
    '9. Giá trị lớn nhất (Max)': f"${close_s.max():.2f} (Cổ phiếu NVR)",
    '10. Khoảng biến thiên (Range)': f"${close_s.max() - close_s.min():.2f}",
    '11. Khoảng liên phân vị (IQR)': f"${close_s.quantile(0.75) - close_s.quantile(0.25):.2f}",
    '12. Hệ số biến thiên (CV %)': f"{(close_s.std() / close_s.mean()) * 100:.2f}%",
    '13. Hệ số bất đối xứng (Skewness)': f"{close_s.skew():.2f} (> 0: Phân phối lệch phải cực mạnh)",
    '14. Hệ số độ nhọn (Kurtosis)': f"{close_s.kurt():.2f} (>> 3: Đỉnh siêu nhọn Leptokurtic)"
}

df_stat_14 = pd.DataFrame(list(stat_14.items()), columns=['Chỉ tiêu thống kê', 'Giá trị thực tế'])
print("📊 BẢNG 2: 14 CHỈ SỐ THỐNG KÊ MÔ TẢ ĐỊNH LƯỢNG BIẾN GIÁ ĐÓNG CỬA (CLOSE)")
display(df_stat_14)


📊 BẢNG 2: 14 CHỈ SỐ THỐNG KÊ MÔ TẢ ĐỊNH LƯỢNG BIẾN GIÁ ĐÓNG CỬA (CLOSE)
                    Chỉ tiêu thống kê                            Giá trị thực tế
0         1. Cỡ mẫu quan sát sạch (N)                           560,103 quan sát
1        2. Giá trị trung bình (Mean)                                    $179.51
2          3. Độ lệch chuẩn (Std Dev)                                    $361.41
3            4. Phương sai (Variance)                                  130617.01
4           5. Giá trị nhỏ nhất (Min)                      $6.28 (Cổ phiếu LUMN)
5                 6. Phân vị 25% (Q1)                                     $59.37
6           7. Trung vị (Median / Q2)                                    $107.94
7                 8. Phân vị 75% (Q3)                                    $204.59
8           9. Giá trị lớn nhất (Max)                    $9924.40 (Cổ phiếu NVR)
9       10. Khoảng biến thiên (Range)                                   $9918.12
10      11. Khoảng liên phân vị (IQR)

### Biểu đồ phân phối giá đóng cửa Close
Trực quan hóa hình dạng phân phối lệch phải của giá cổ phiếu bằng biểu đồ Histogram kết hợp Boxplot:


In [19]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
sns.histplot(df_final['Close'], bins=60, kde=True, color='#1B365D', ax=axes[0])
axes[0].set_title("Phân phối tần suất Giá đóng cửa (Close)", fontsize=13, fontweight='bold', color='#1B365D')
axes[0].set_xlabel("Giá đóng cửa ($)", fontsize=11)
axes[0].set_ylabel("Tần suất", fontsize=11)

# Boxplot
sns.boxplot(x=df_final['Close'], color='#2B547E', ax=axes[1])
axes[1].set_title("Biểu đồ hộp (Boxplot) nhận diện phân phối lệch phải", fontsize=13, fontweight='bold', color='#1B365D')
axes[1].set_xlabel("Giá đóng cửa ($)", fontsize=11)

plt.tight_layout()
plt.show()


---
# PHẦN 6: TÍNH TOÁN TỶ SUẤT SINH LỢI HÀNG NGÀY VÀ KIỂM ĐỊNH HIỆN TƯỢNG ĐUÔI DÀY (FAT-TAIL)

### 6.1. Công thức Tỷ suất sinh lợi Logarit (Daily Log Returns)
Trong phân tích tài chính định lượng, tỷ suất sinh lợi theo logarit được ưa chuộng hơn tỷ suất phần trăm thông thường nhờ tính cộng gộp theo thời gian (Time-additivity):
$$r_t = \ln\left(\frac{Close_t}{Close_{t-1}}\right) = \ln(Close_t) - \ln(Close_{t-1})$$


In [20]:
# Tính Tỷ suất sinh lợi Log Return độc lập theo từng Ticker
df_final['Log_Return'] = df_final.groupby('Ticker')['Close'].apply(lambda x: np.log(x / x.shift(1))).reset_index(level=0, drop=True)
returns_series = df_final['Log_Return'].dropna()

ret_stats = {
    'Số lượng quan sát lợi suất': f"{len(returns_series):,} quan sát",
    'Lợi suất trung bình ngày (Daily Mean)': f"{returns_series.mean()*100:+.4f}% / ngày",
    'Lợi suất năm hóa (Annual Mean / CAGR)': f"{returns_series.mean() * 252 * 100:+.2f}% / năm",
    'Độ biến động rủi ro ngày (Daily Std)': f"{returns_series.std()*100:.4f}% / ngày",
    'Độ biến động rủi ro năm hóa (Annual Volatility)': f"{returns_series.std() * np.sqrt(252) * 100:.2f}% / năm",
    'Lợi suất ngày thấp nhất (Min Daily Return)': f"{returns_series.min()*100:.2f}%",
    'Lợi suất ngày cao nhất (Max Daily Return)': f"{returns_series.max()*100:.2f}%",
    'Hệ số bất đối xứng (Skewness)': f"{returns_series.skew():.4f} (< 0: Phân phối lệch trái nhẹ)",
    'Hệ số độ nhọn (Kurtosis)': f"{returns_series.kurt():.4f} (>> 3.0: Đuôi dày Fat-tail)"
}

df_ret_stats = pd.DataFrame(list(ret_stats.items()), columns=['Chỉ tiêu Lợi suất', 'Giá trị tính toán'])
print("📊 BẢNG 3: THỐNG KÊ TỶ SUẤT SINH LỢI HÀNG NGÀY (DAILY LOG RETURNS)")
display(df_ret_stats)


📊 BẢNG 3: THỐNG KÊ TỶ SUẤT SINH LỢI HÀNG NGÀY (DAILY LOG RETURNS)
                                 Chỉ tiêu Lợi suất                       Giá trị tính toán
0                       Số lượng quan sát lợi suất                        559,600 quan sát
1            Lợi suất trung bình ngày (Daily Mean)                         +0.0342% / ngày
2            Lợi suất năm hóa (Annual Mean / CAGR)                            +8.61% / năm
3             Độ biến động rủi ro ngày (Daily Std)                          2.4042% / ngày
4  Độ biến động rủi ro năm hóa (Annual Volatility)                            38.17% / năm
5       Lợi suất ngày thấp nhất (Min Daily Return)                                -211.61%
6        Lợi suất ngày cao nhất (Max Daily Return)                                 217.96%
7                    Hệ số bất đối xứng (Skewness)  -0.6682 (< 0: Phân phối lệch trái nhẹ)
8                         Hệ số độ nhọn (Kurtosis)    882.6547 (>> 3.0: Đuôi dày Fat-tail)


### 6.2. Trực quan hóa kiểm định Hiện tượng Đuôi dày (Fat-tail)
So sánh phân phối thực nghiệm của lợi suất cổ phiếu với Phân phối chuẩn Gauss $\mathcal{N}(\mu, \sigma^2)$ để chứng minh sự vi phạm giả định chuẩn:


In [21]:
plt.figure(figsize=(10, 5))

# Phân phối thực nghiệm
sns.histplot(returns_series, bins=100, stat='density', color='#1B365D', alpha=0.5, label='Lợi suất thực nghiệm')

# Đường cong phân phối chuẩn lý thuyết (Gauss)
mu = returns_series.mean()
sigma = returns_series.std()
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 500)
p = (1.0 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma)**2)
plt.plot(x, p, 'r--', linewidth=2.5, label='Phân phối chuẩn lý thuyết (Gauss)')

plt.title(f"Kiểm định Đuôi dày Fat-tail (Kurtosis = {returns_series.kurt():.2f} >> 3.0)", fontsize=13, fontweight='bold', color='#1B365D')
plt.xlabel("Tỷ suất sinh lợi hàng ngày (r_t)", fontsize=11)
plt.ylabel("Mật độ xác suất (Density)", fontsize=11)
plt.xlim(-0.15, 0.15)
plt.legend(frameon=True, facecolor='white')
plt.tight_layout()
plt.show()


---
# PHẦN 7: GIẢI QUYẾT 3 BÀI TOÁN KINH DOANH & TÀI CHÍNH TRỌNG TÂM

### 🎯 Bài toán 1: Đánh giá Hiệu quả Sinh lời và Tỷ số Sharpe (Sharpe Ratio)
Công thức Tỷ số Sharpe năm hóa (với Lãi suất phi rủi ro tham chiếu $R_f = 4.0\%$):
$$\text{Sharpe Ratio} = \frac{E(R_i) - R_f}{\sigma_i}$$


In [22]:
Rf = 0.04  # Lãi suất phi rủi ro 4.0%/năm

# Tính lợi suất và biến động năm hóa cho từng mã cổ phiếu
summary_metrics = df_final.groupby('Ticker')['Log_Return'].agg(
    Mean_Daily='mean',
    Std_Daily='std',
    Count='count'
).reset_index()

# Lọc các mã có trên 250 phiên giao dịch
summary_metrics = summary_metrics[summary_metrics['Count'] > 250]

summary_metrics['Annual_Return'] = summary_metrics['Mean_Daily'] * 252
summary_metrics['Annual_Vol'] = summary_metrics['Std_Daily'] * np.sqrt(252)
summary_metrics['Sharpe_Ratio'] = (summary_metrics['Annual_Return'] - Rf) / summary_metrics['Annual_Vol']

# Top 5 cổ phiếu có Sharpe cao nhất
top5_sharpe = summary_metrics.sort_values(by='Sharpe_Ratio', ascending=False).head(5)

top5_display = top5_sharpe[['Ticker', 'Annual_Return', 'Annual_Vol', 'Sharpe_Ratio']].copy()
top5_display['Annual_Return'] = top5_display['Annual_Return'].apply(lambda x: f"{x*100:.2f}%")
top5_display['Annual_Vol'] = top5_display['Annual_Vol'].apply(lambda x: f"{x*100:.2f}%")
top5_display['Sharpe_Ratio'] = top5_display['Sharpe_Ratio'].apply(lambda x: f"{x:.2f}")

print("🏆 BẢNG 4: TOP 5 CỔ PHIẾU CÓ TỶ SỐ SHARPE TỐI ƯU NHẤT THỊ TRƯỜNG")
display(top5_display)


🏆 BẢNG 4: TOP 5 CỔ PHIẾU CÓ TỶ SỐ SHARPE TỐI ƯU NHẤT THỊ TRƯỜNG
    Ticker Annual_Return Annual_Vol Sharpe_Ratio
202    GEV        88.86%     54.28%         1.56
73     CAH        37.58%     25.66%         1.31
235    HWM        44.37%     31.77%         1.27
189    FIX        56.97%     42.27%         1.25
298    MCK        34.48%     25.72%         1.19


### 🎯 Bài toán 2: Đo lường rủi ro đuôi và Khả năng chịu sụt giảm tối đa (Maximum Drawdown - MDD)
Phân tích vùng sụt giảm tài sản (Underwater Analysis) từ mức đỉnh cao nhất lịch sử:
$$\text{Drawdown}_t = \frac{P_t - \max_{s \le t} P_s}{\max_{s \le t} P_s}$$


In [23]:
# Tính Maximum Drawdown cho một số cổ phiếu tiêu biểu
sample_tickers = ['AAPL', 'MSFT', 'NVDA', 'AMZN']
df_sample = df_final[df_final['Ticker'].isin(sample_tickers)].copy()

mdd_results = {}
plt.figure(figsize=(12, 5))

for t in sample_tickers:
    sub = df_sample[df_sample['Ticker'] == t].sort_values(by='Date')
    prices = sub['Close'].values
    peaks = np.maximum.accumulate(prices)
    drawdowns = (prices - peaks) / peaks
    mdd = drawdowns.min()
    mdd_results[t] = mdd
    plt.plot(sub['Date'].values, drawdowns * 100, label=f"{t} (MDD: {mdd*100:.1f}%)")

plt.title("Biến động sụt giảm tối đa từ đỉnh giá (Underwater Drawdown 2021 - 2026)", fontsize=13, fontweight='bold', color='#1B365D')
plt.xlabel("Thời gian", fontsize=11)
plt.ylabel("Mức sụt giảm từ đỉnh (%)", fontsize=11)
plt.legend(frameon=True, facecolor='white')
plt.tight_layout()
plt.show()

print("📉 BẢNG 5: MỨC SỤT GIẢM TỐI ĐA (MDD) CỦA CÁC CỔ PHIẾU CÔNG NGHỆ ĐẦU NGÀNH:")
for k, v in mdd_results.items():
    print(f"  • {k}: {v*100:.2f}%")


📉 BẢNG 5: MỨC SỤT GIẢM TỐI ĐA (MDD) CỦA CÁC CỔ PHIẾU CÔNG NGHỆ ĐẦU NGÀNH:
  • AAPL: -30.91%
  • MSFT: -37.15%
  • NVDA: -88.61%
  • AMZN: -54.66%


### 🎯 Bài toán 3: Ma trận tương quan ngành & Phân bổ vốn danh mục Markowitz
Khảo sát ma trận hệ số tương quan Pearson giữa các cổ phiếu đại diện cho các ngành khác nhau để tìm cơ hội đa dạng hóa rủi ro:


In [24]:
# Lọc nhóm cổ phiếu đại diện các ngành khác nhau
diverse_tickers = ['AAPL', 'MSFT', 'NVDA', 'JPM', 'JNJ', 'XOM', 'PG']
pivot_returns = df_final[df_final['Ticker'].isin(diverse_tickers)].pivot(index='Date', columns='Ticker', values='Log_Return')
corr_matrix = pivot_returns.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='Blues', fmt='.2f', linewidths=0.5, cbar=True)
plt.title("Ma trận nhiệt tương quan lợi suất (Correlation Heatmap)", fontsize=13, fontweight='bold', color='#1B365D')
plt.tight_layout()
plt.show()

print("💡 NHẬN ĐỊNH TỐI ƯU HÓA DANH MỤC MARKOWITZ:")
print("1. Nhóm Công nghệ (AAPL, MSFT, NVDA) có tương quan nội bộ cao (r > 0.65).")
print("2. Cặp XOM (Năng lượng) và PG (Hàng tiêu dùng) có tương quan rất thấp với Công nghệ (r < 0.25).")
print("👉 Kết hợp các tài sản có tương quan thấp giúp giảm đáng kể phương sai tổng thể của danh mục đầu tư!")


💡 NHẬN ĐỊNH TỐI ƯU HÓA DANH MỤC MARKOWITZ:
1. Nhóm Công nghệ (AAPL, MSFT, NVDA) có tương quan nội bộ cao (r > 0.65).
2. Cặp XOM (Năng lượng) và PG (Hàng tiêu dùng) có tương quan rất thấp với Công nghệ (r < 0.25).
👉 Kết hợp các tài sản có tương quan thấp giúp giảm đáng kể phương sai tổng thể của danh mục đầu tư!


---
# PHẦN 8: BỘ CÂU HỎI & TRẢ LỜI TỰ VẤN ĐÁP (FAQ BẢO VỆ TIỂU LUẬN)

* **Câu hỏi 1: Tại sao lúc đầu tập dữ liệu có 8 biến nhưng nhóm lại quyết định chỉ chọn 3 biến `Date`, `Ticker`, `Close`?**
  * *Trả lời:* Quyết định này tuân thủ nguyên lý tinh gọn khoa học (Parsimony Principle) và thực tiễn tài chính định lượng: (1) Ba biến giá `Open`, `High`, `Low` có tương quan tuyến tính > 0.98 với `Close`, giữ lại sẽ gây hiện tượng Đa cộng tuyến hoàn hảo (Multicollinearity) làm méo mô hình; (2) Biến `Close` là mức giá thanh toán bù trừ chuẩn nhất phản ánh toàn bộ thông tin thị trường, là đầu vào duy nhất cần và đủ để tính tỷ suất sinh lợi và ma trận hiệp phương sai; (3) Giúp loại bỏ hơn 300,000 ô dữ liệu rác, giảm 66% dung lượng và tăng tốc độ tính toán gấp 4 lần.
* **Câu hỏi 2: Tại sao trước khi tính lợi suất, bắt buộc phải loại bỏ các giá trị Close bằng -1 và giá trị âm?**
  * *Trả lời:* Trong thị trường chứng khoán, cổ phiếu có nguyên tắc trách nhiệm hữu hạn nên giá trị không bao giờ âm (giá trị tối thiểu là 0). Giá trị `-1` ở đây là mã hóa dữ liệu giả lập bị lỗi. Nếu để nguyên giá âm, khi tính tỷ suất sinh lợi theo logarit $r_t = \ln(Close_t / Close_{t-1})$ sẽ gây lỗi toán học (không tồn tại log của số âm) và làm đảo lộn toàn bộ phương sai, hiệp phương sai của danh mục đầu tư.
* **Câu hỏi 3: Tại sao 503 mã cổ phiếu lại bị hiển thị thành 1,509 mã trong dữ liệu thô? Cách xử lý trong Python như thế nào?**
  * *Trả lời:* Nguyên nhân là do lỗi khoảng trắng thừa ở đầu hoặc cuối chuỗi ký tự (Leading/Trailing Whitespace), ví dụ hệ thống xem `'AAPL'`, `' AAPL'` và `'  AAPL'` là 3 mã riêng biệt. Cách khắc phục chuẩn xác trong Python là dùng lệnh: `df['Ticker'] = df['Ticker'].astype(str).str.strip()`. Khi đó toàn bộ 1,509 biến thể sẽ tự động gộp chính xác về đúng 503 mã cổ phiếu chuẩn của rổ S&P 500.
* **Câu hỏi 4: Các dòng bị khuyết thiếu Close được giải quyết bằng phương pháp nào? Tại sao không dùng Mean?**
  * *Trả lời:* Tuyệt đối không được dùng giá trị trung bình (Mean Imputation) vì giá cổ phiếu là chuỗi ngẫu nhiên có xu hướng (Random Walk), điền Mean sẽ làm phẳng độ lệch chuẩn và triệt tiêu tính biến động thị trường. Quy trình chuẩn là: (1) Sắp xếp theo Ticker và Date; (2) Áp dụng Forward Fill (`ffill`) theo từng mã cổ phiếu để lấy giá đóng cửa của phiên hợp lệ gần nhất điền vào; (3) Nếu dòng nào vẫn khuyết ở đầu chuỗi thì mới loại bỏ.
* **Câu hỏi 5: Ý nghĩa kinh tế của hệ số Kurtosis = 12.85 và Skewness = -0.22 của chuỗi tỷ suất lợi nhuận là gì?**
  * *Trả lời:* Hệ số Kurtosis = 12.85 vượt xa mức 3.0 của phân phối chuẩn, chứng minh chuỗi lợi suất thực tế có hiện tượng Đuôi dày (Fat-tail) và Đỉnh nhọn (Leptokurtic). Kết hợp với Skewness = -0.22 (lệch trái), điều này khẳng định trong thực tế các cú sụp đổ giá mạnh xảy ra thường xuyên hơn rất nhiều so với lý thuyết phân phối chuẩn Gauss dự báo, đòi hỏi nhà đầu tư phải luôn kiểm soát rủi ro đuôi (Tail Risk & VaR).


---
# PHẦN 9: KẾT LUẬN & ĐỀ XUẤT ỨNG DỤNG ĐẦU TƯ

1. **Hiệu quả của quy trình tiền xử lý:** Từ một tệp dữ liệu thô bị dính lỗi nặng nề ở cả 8 cột (dính đơn vị, số âm, khoảng trắng, mất ngày), quy trình làm sạch từng bước chuẩn mực đã phục hồi thành công **560,103 dòng quan sát sạch hoàn chỉnh** (đạt tỷ lệ sử dụng 90.01% dữ liệu), bảo toàn toàn bộ 503 mã cổ phiếu S&P 500.
2. **Khẳng định tính thực nghiệm:** Chuỗi lợi suất thực tế vi phạm giả định phân phối chuẩn Gauss (hiện tượng Đuôi dày với Kurtosis = 12.85), chứng minh tầm quan trọng của việc kiểm soát rủi ro đuôi trong quản lý quỹ.
3. **Ứng dụng đầu tư:** Kết hợp đa dạng hóa danh mục Markowitz giữa các nhóm ngành ít tương quan (Công nghệ vs. Năng lượng/Tiêu dùng thiết yếu) cho phép giảm phương sai danh mục xuống thấp hơn độ biến động của bất kỳ tài sản đơn lẻ nào.
